<a href="https://colab.research.google.com/github/aynuraaxmedova345-max/burgut/blob/main/module-6/Simple_RAG_RAG_EDU_1_COMPLETE_All_Parts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Simple RAG — Educational Document Question-Answering System
**System designation: RAG-EDU-1** · Built strictly according to the Technical Specification (BePro Academy, 2026)

---

## Project Overview

**RAG (Retrieval-Augmented Generation)** is an approach in which a large language model (LLM) answers a question **not from its own internal knowledge**, but from document fragments retrieved from *your* knowledge base.

Think of it as an **open-book exam**: before answering, the system finds the relevant "page" from your documents and hands it to the model.

### Why do we need this?

An LLM does not know the content of your private documents. If you ask it about them anyway, it tends to produce a confident but **invented** answer — this is called a *hallucination*. RAG eliminates the problem by forcing the model to answer **only** from the retrieved fragments, and to say *"the information is unavailable"* when the documents do not contain the answer.

### System architecture (per the Technical Specification, Appendix B)

The system consists of **two subsystems** connected through a shared **vector database**:

**Stage 1 — Document indexing** *(performed once)*
```
Documents → Split into fragments (chunks) → Vectorize (embeddings) → Store in vector database
```

**Stage 2 — Query processing** *(performed on each request)*
```
Question → Vectorize question → Retrieve nearest fragments → Assemble prompt → Generate answer → Return answer + sources
```

### Notebook roadmap (5 parts)

| Part | Content | TS reference |
|---|---|---|
| **1 (this part)** | Environment setup, OpenAI configuration, document loading & verification | §4.3, §5 stages 1–2 |
| 2 | Chunking + embedding generation | §4.2, Appendix A |
| 3 | ChromaDB storage + similarity search (retrieval) | §4.2 |
| 4 | RAG prompt + answer generation + source attribution | §4.2 |
| 5 | Acceptance testing + compliance checklist | §6 |

### Requirements to run this notebook

- A web browser and a Google account (Google Colab) — **no local installation needed** (TS §4.1)
- A valid **OpenAI API key** (instructions below — you will insert your own key securely)
- Your `.txt` documents (this notebook uses **only your uploaded files** as the knowledge base)


---
# Part 1 · Section 1.1 — Environment Setup: Installing Libraries

### Theory

Python programs rarely work alone — they rely on **libraries**: pre-written, tested packages of code. Per TS §4.3, this system uses exactly two external libraries:

| Library | What it does in our system |
|---|---|
| `openai` | Official OpenAI SDK. Gives us access to **two** models: the *embedding* model (`text-embedding-3-small`, turns text into numeric vectors) and the *generation* model (`gpt-4o-mini`, writes the final answer). |
| `chromadb` | An open-source **vector database**. It stores each text fragment together with its vector, and can quickly find the fragments *closest in meaning* to a query vector. |

### Why this step is needed

Google Colab machines start "fresh" every session — they include Python and many scientific packages, but **not** `chromadb`, and the pre-installed `openai` version may be outdated. Installing explicitly guarantees a reproducible environment (TS §6, check 5: *reproducibility*).

### How it works internally

`pip` is Python's package manager. It downloads packages from PyPI (the Python Package Index) and installs them into the Colab runtime. The `-q` flag ("quiet") hides the long installation log so the notebook stays readable. The leading `!` tells Colab: *run this as a shell command, not Python code*.


In [1]:
# ── 1.1 Install required libraries ─────────────────────────────────
# -q  : quiet mode (less noise in the output)
# -U  : upgrade to the latest version (avoids deprecated APIs)
!pip install -q -U openai chromadb

# Verify the installation by importing the packages and printing versions
import openai
import chromadb

print(f"openai   version: {openai.__version__}")
print(f"chromadb version: {chromadb.__version__}")
print("\n✅ Libraries installed and imported successfully.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 k

### Code explanation

- `!pip install -q -U openai chromadb` — installs both libraries in one command; `-U` ensures we get the **latest OpenAI SDK (v1.x)**, whose interface differs from the old deprecated v0.x style.
- The `import` lines double as a **verification step**: if installation failed, the import raises an error immediately, instead of surprising us later.

### Example output

```
openai   version: 1.9x.x
chromadb version: 1.x.x

✅ Libraries installed and imported successfully.
```

### Conclusion

The runtime now contains everything the system needs. If you restart the Colab runtime (`Runtime → Restart session`), you must re-run this cell — Colab machines do not remember installed packages between sessions.


---
# Part 1 · Section 1.2 — OpenAI API Configuration (Secure)

### Theory

An **API key** is your personal password to OpenAI's models. Every request the system makes (embedding a chunk, generating an answer) is authenticated — and billed — with this key.

### Why this step is needed

Two rules of professional engineering:

1. **Never hardcode a key in a notebook.** Anyone you share the notebook with would get your key (and could spend your money).
2. **Never print a key.** Even in your own output cells.

This notebook therefore reads your key from a **secure store** and only ever confirms *that a key exists*, never *what it is*.

### How it works internally

We try two secure sources, in order:

1. **Google Colab Secrets** (recommended) — Colab's built-in encrypted key store. The key lives in your Google account, not in the notebook file.
2. **Environment variable** `OPENAI_API_KEY` — the standard fallback that also works when running locally (TS §4.3 permits local execution).

---
### 🔑 WHERE TO INSERT YOUR API KEY (do this once, before running the cell below)

**Recommended — Colab Secrets:**
1. Click the **🔑 key icon** in the left sidebar of Colab ("Secrets").
2. Click **"+ Add new secret"**.
3. Name: `OPENAI_API_KEY` · Value: *paste your key here*.
4. Toggle **"Notebook access" → ON**.

**Alternative — environment variable (e.g., when running locally):** uncomment the marked line in the cell below and paste your key there **only on your private machine, never in a shared notebook**.


In [3]:
# ── 1.2 Configure OpenAI API access ────────────────────────────────
import os
from openai import OpenAI

# ── ALTERNATIVE (local runs only): uncomment and insert your key ──
# os.environ["OPENAI_API_KEY"] = "sk-...PASTE-YOUR-KEY-HERE..."

def get_api_key() -> str:
    """Return the OpenAI API key from Colab Secrets or the environment.

    Search order:
      1. Google Colab Secrets  (secret name: OPENAI_API_KEY)
      2. Environment variable  OPENAI_API_KEY

    Raises:
        RuntimeError: if no key is found in either location.
    """
    # Source 1: Google Colab Secrets (only available inside Colab)
    try:
        from google.colab import userdata  # type: ignore
        key = userdata.get("OPENAI_API_KEY")
        if key:
            print("🔐 API key loaded from Google Colab Secrets.")
            return key
    except Exception:
        pass  # not running in Colab, or the secret is not set

    # Source 2: environment variable
    key = os.environ.get("OPENAI_API_KEY", "")
    if key:
        print("🔐 API key loaded from the OPENAI_API_KEY environment variable.")
        return key

    raise RuntimeError(
        "No OpenAI API key found.\n"
        "→ In Colab: open the 🔑 'Secrets' panel (left sidebar), add a secret "
        "named OPENAI_API_KEY, and enable 'Notebook access'.\n"
        "→ Locally: set the OPENAI_API_KEY environment variable."
    )

# Create a single OpenAI client used by the whole system
client = OpenAI(api_key=get_api_key())
print("✅ OpenAI client is configured. (The key itself is never displayed.)")


🔐 API key loaded from Google Colab Secrets.
✅ OpenAI client is configured. (The key itself is never displayed.)


### Code explanation

- `get_api_key()` — a small **reusable function** with a docstring and a clear search order. If no key is found, it raises a helpful error telling you exactly what to do, instead of failing mysteriously later.
- `OpenAI(api_key=...)` — the modern **v1.x SDK** pattern: one `client` object is created once and reused for every embedding and generation call in Parts 2–4.
- Note what the cell prints: *where* the key came from — never the key itself.

### Example output

```
🔐 API key loaded from Google Colab Secrets.
✅ OpenAI client is configured. (The key itself is never displayed.)
```

### Conclusion

The system can now authenticate with OpenAI securely. Next we define the system's central configuration, then load your documents.


---
# Part 1 · Section 1.3 — Central System Configuration

### Theory & why this step is needed

The Technical Specification (**Appendix A**) defines six configurable parameters that control retrieval quality and API cost. Professional systems keep all such parameters in **one place**, not scattered through the code — so they can be changed *one at a time* and re-tested, exactly as the TS instructs.

Two parameters are **fixed by the TS** and must not be changed:
- the embedding model — `text-embedding-3-small`
- the generation model — `gpt-4o-mini`

⚠️ **Critical TS rule:** the embedding model used for *indexing* and for *queries* must be **the same**, otherwise retrieval will not work. Keeping the model name in a single config constant makes violating this rule impossible.


In [4]:
# ── 1.3 Central configuration (TS Appendix A) ──────────────────────
CONFIG: dict = {
    # Chunking (used in Part 2)
    "chunk_size": 600,        # characters per fragment (TS range: 300–800)
    "chunk_overlap": 80,      # shared characters between adjacent chunks (TS: 50–100)

    # Retrieval (used in Part 3)
    "top_k": 4,               # fragments retrieved per query (TS: 3–5)

    # Models (FIXED by the Technical Specification — do not change)
    "embedding_model": "text-embedding-3-small",
    "generation_model": "gpt-4o-mini",

    # Generation (used in Part 4)
    "temperature": 0,         # 0 = factual mode, no "creativity" (TS Appendix A)
}

print("System configuration (TS Appendix A):\n")
for parameter, value in CONFIG.items():
    print(f"  {parameter:<18} = {value}")


System configuration (TS Appendix A):

  chunk_size         = 600
  chunk_overlap      = 80
  top_k              = 4
  embedding_model    = text-embedding-3-small
  generation_model   = gpt-4o-mini
  temperature        = 0


### Conclusion

Every later part of the notebook reads its parameters from `CONFIG`. To experiment (e.g., a different chunk size), change **one value here**, re-run indexing, and re-check retrieval quality — per TS Appendix A.


---
# Part 1 · Section 1.4 — Loading the Knowledge-Base Documents

### Theory

The knowledge base of this system is **your uploaded `.txt` files** — nothing else. The TS (§3) defines the input as text documents; this notebook automatically detects **every** `.txt` file you upload and loads it into memory.

### Why this step is needed

Before we can split, vectorize, or search anything, the raw text must be read from disk into Python strings — **correctly**. Real-world text files hide two classic traps:

1. **Encoding.** Uzbek text uses characters like `o‘`, `g‘`, `ʼ` that only decode correctly as **UTF-8**. Reading with the wrong encoding produces garbage (mojibake). We use `utf-8-sig`, which also transparently removes a BOM (byte-order mark) if a file was saved on Windows.
2. **Line endings.** Files saved on Windows end lines with `\r\n` (CRLF). We normalize these to `\n` so that chunking in Part 2 behaves identically for every file.

### How it works internally

1. When this cell runs in Colab, it opens a **file-upload dialog** — select your `.txt` files (e.g., `Murdalar_gapirmaydilar.txt` and `Ikki_eshik_orasi__2_.txt`).
2. If `.txt` files are already present in the working directory (e.g., re-running the notebook, or running locally), they are detected automatically and the dialog is skipped.
3. Each file is read as UTF-8, normalized, and stored in a dictionary `documents = {filename: text}` — the single source of data for all later parts.


In [6]:
# ── 1.4 Upload and load all .txt documents ─────────────────────────
from pathlib import Path

def find_txt_files(directory: str = ".") -> list[Path]:
    """Return a sorted list of all .txt files in the given directory."""
    return sorted(Path(directory).glob("*.txt"))

def upload_documents_if_needed() -> list[Path]:
    """Ensure .txt files are available; open Colab's upload dialog if not.

    Returns:
        A list of paths to all detected .txt files.
    """
    txt_files = find_txt_files()
    if txt_files:
        print(f"📁 Found {len(txt_files)} .txt file(s) already in the working directory.")
        return txt_files

    try:
        from google.colab import files  # type: ignore
        print("⬆️ Please select your .txt knowledge-base files in the dialog below…")
        files.upload()  # writes the selected files into the working directory
    except ImportError:
        pass  # not running in Colab — files must already be on disk

    return find_txt_files()

def load_documents(paths: list[Path]) -> dict[str, str]:
    """Read every .txt file into memory as normalized UTF-8 text.

    Args:
        paths: Paths of the .txt files to load.

    Returns:
        Mapping of {filename: document text}.

    Raises:
        RuntimeError: if no .txt files were provided.
    """
    if not paths:
        raise RuntimeError(
            "No .txt files found. Please run this cell again and upload "
            "your knowledge-base documents."
        )

    documents: dict[str, str] = {}
    for path in paths:
        text = path.read_text(encoding="utf-8-sig")   # handles BOM safely
        text = text.replace("\r\n", "\n").strip()     # normalize line endings
        documents[path.name] = text
    return documents

document_paths = upload_documents_if_needed()
documents = load_documents(document_paths)
print(f"\n✅ Loaded {len(documents)} document(s) into memory.")


📁 Found 1 .txt file(s) already in the working directory.

✅ Loaded 1 document(s) into memory.


### Code explanation

- **Three small functions instead of one big block** — each does one job (find files / upload / load), has a docstring and type hints, and can be reused or tested independently.
- `utf-8-sig` — decodes UTF-8 *and* silently strips a Windows BOM if present; for a BOM-less file it behaves exactly like plain `utf-8`.
- `documents` is a plain `{filename: text}` dictionary — keeping the **filename** attached to the text now is what makes **source attribution** possible in Part 4 (TS §6, check 4).


---
# Part 1 · Section 1.5 — Verification of Document Loading

### Why this step is needed

The TS (§6) demands that the system be verified stage by stage. Before we spend money on embedding API calls in Part 2, we prove that **every** document was loaded, is non-empty, and decoded correctly. The preview below is the human check: if the Uzbek characters (`o‘`, `g‘`, `ʼ`) display correctly, the encoding step worked.


In [7]:
# ── 1.5 Verify document loading ─────────────────────────────────────
def verify_documents(docs: dict[str, str], preview_chars: int = 300) -> None:
    """Print a loading report and run basic sanity checks on each document.

    Checks performed:
      * the document set is not empty
      * every document contains non-empty text

    Args:
        docs: Mapping of {filename: document text}.
        preview_chars: Number of characters to show as a content preview.
    """
    assert docs, "Verification failed: no documents were loaded."

    print(f"{'='*70}")
    print(f"DOCUMENT LOADING REPORT — {len(docs)} file(s)")
    print(f"{'='*70}")

    for index, (filename, text) in enumerate(docs.items(), start=1):
        assert text, f"Verification failed: '{filename}' is empty."
        size_kb = len(text.encode('utf-8')) / 1024
        word_count = len(text.split())

        print(f"\n📄 Document {index}: {filename}")
        print(f"   Size        : {size_kb:,.1f} KB")
        print(f"   Characters  : {len(text):,}")
        print(f"   Words       : {word_count:,}")
        print(f"   Preview     : {text[:preview_chars]!r} …")

    print(f"\n{'='*70}")
    print("✅ VERIFICATION PASSED — every document is loaded and non-empty.")
    print(f"{'='*70}")

verify_documents(documents)


DOCUMENT LOADING REPORT — 1 file(s)

📄 Document 1: Ikki eshik orasi (2).txt
   Size        : 1,116.1 KB
   Characters  : 1,074,925
   Words       : 141,535
   Preview     : '«Rost bilan yolg‘onning o‘rtasi – to‘rt enlik», degan gap bor.\nQiziq, nega endi oz emas, ko‘p emas, to‘rt enlik?\nGap shundaki, ko‘z bilan quloqning orasi – to‘rt enlik ekan.\nEshitganingga emas, ko‘rganingga ishon.\nMaqsad – shu. Bu kitobdagi ko‘p odamlarni o‘zim ko‘rganman.\nKo‘plari bilan gaplashganm' …

✅ VERIFICATION PASSED — every document is loaded and non-empty.


### Example output (with the two provided documents)

```
======================================================================
DOCUMENT LOADING REPORT — 2 file(s)
======================================================================

📄 Document 1: Ikki_eshik_orasi__2_.txt
   Size        : ~1,120.0 KB
   Characters  : ~1,100,000
   Words       : ~134,000
   Preview     : '«Rost bilan yolg‘onning o‘rtasi – to‘rt enlik», degan gap bor. …'

📄 Document 2: Murdalar_gapirmaydilar.txt
   Size        : ~144.0 KB
   Characters  : ~148,000
   Words       : ~18,000
   Preview     : 'Butun olamlar Parvardigori Allohga behad hamd bo‘lsin …'

======================================================================
✅ VERIFICATION PASSED — every document is loaded and non-empty.
======================================================================
```

### Conclusion — Part 1 complete ✅

| Requirement (TS / prompt) | Status |
|---|---|
| Libraries installed (`openai`, `chromadb` only — TS §4.3) | ✅ |
| Latest OpenAI SDK (v1.x), no deprecated APIs | ✅ |
| API key via Colab Secrets / env var — never hardcoded, never printed | ✅ |
| Central configuration matching TS Appendix A | ✅ |
| Only the user's uploaded `.txt` files used — no demo documents created | ✅ |
| Auto-detection of every uploaded `.txt` file | ✅ |
| Filename, size, file count, and content preview displayed | ✅ |
| Loading verified for every document | ✅ |

**Next — Part 2:** splitting the documents into overlapping chunks and generating embeddings with `text-embedding-3-small`.


---
---
# Part 2 · Section 2.1 — Document Preprocessing: Chunking

### Theory

Our documents are far too large to hand to a language model whole — one of them is over a million characters. And even if they fit, it would be wasteful: to answer *one* question, the model only needs the few paragraphs that actually discuss it.

**Chunking** solves this: we cut every document into small **fragments (chunks)** of a fixed size. Later, the system searches *among the chunks* and sends the model only the 3–5 most relevant ones (TS §4.2).

The size of a chunk is a trade-off:

| Chunk size | Effect |
|---|---|
| Too small (e.g., 100 chars) | A chunk loses its context — a sentence fragment means little on its own |
| Too large (e.g., 5,000 chars) | Retrieval becomes imprecise, and irrelevant text "dilutes" the prompt |
| **TS range: 300–800 chars** | Roughly a paragraph — one coherent thought per chunk |

### Why overlap is needed

Imagine an important sentence sitting **exactly on the border** between chunk 17 and chunk 18 — half in each. Neither chunk contains the full thought, so *neither* would be retrieved for a question about it.

**Overlap** is the fix: each new chunk starts slightly *before* the previous one ended, so adjacent chunks **share** a strip of text (TS: 50–100 characters). Any sentence near a boundary appears **whole in at least one** of the two chunks. The TS calls this *"protecting answers that straddle a boundary."*

```
Chunk 1:  [aaaaaaaaaaaaaaaaaaaaaaXXX]
Chunk 2:                       [XXXbbbbbbbbbbbbbbbbbbbb]
                                ↑ overlap — shared text
```

### How it works internally

Our chunker walks through the text with a sliding window:

1. Take a window of `chunk_size` characters (600, from `CONFIG`).
2. **Snap the cut to a word boundary** — if the window would slice a word in half, pull the cut back to the last space or line break. Cutting a word in the middle damages both its meaning and its embedding.
3. Save the chunk together with its **source filename** and a unique **chunk id** — this metadata is what powers source attribution in Part 4.
4. Step the window forward by `chunk_size − overlap`, so the next chunk re-includes the last ~80 characters.
5. Repeat until the end of the document.


In [8]:
# ── 2.1 Split documents into overlapping chunks ────────────────────
def chunk_text(text: str, chunk_size: int, overlap: int) -> list[str]:
    """Split text into chunks of ~chunk_size characters with overlap.

    Cuts are snapped back to the nearest word boundary (space or line
    break) so that no word is ever sliced in half.

    Args:
        text: The full document text.
        chunk_size: Target size of one chunk, in characters.
        overlap: Number of characters shared by adjacent chunks.

    Returns:
        List of chunk strings, in document order.
    """
    if overlap >= chunk_size:
        raise ValueError("overlap must be smaller than chunk_size")

    chunks: list[str] = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))

        # Snap the cut to a word boundary (unless we are at the very end)
        if end < len(text):
            boundary = max(text.rfind(" ", start, end),
                           text.rfind("\n", start, end))
            if boundary > start:          # found a safe place to cut
                end = boundary

        chunk = text[start:end].strip()
        if chunk:                          # skip fragments that are pure whitespace
            chunks.append(chunk)

        if end >= len(text):
            break
        start = max(end - overlap, start + 1)   # slide back by `overlap`
    return chunks


def build_chunk_records(docs: dict[str, str],
                        chunk_size: int,
                        overlap: int) -> list[dict]:
    """Chunk every document and attach metadata to each fragment.

    Args:
        docs: Mapping of {filename: document text}.
        chunk_size: Target chunk size in characters.
        overlap: Overlap between adjacent chunks in characters.

    Returns:
        A list of records: {"id", "text", "source", "chunk_index"}.
        The "source" field enables source attribution (TS §6, check 4).
    """
    records: list[dict] = []
    for filename, text in docs.items():
        for i, chunk in enumerate(chunk_text(text, chunk_size, overlap)):
            records.append({
                "id": f"{filename}::chunk_{i:04d}",   # unique, human-readable id
                "text": chunk,
                "source": filename,
                "chunk_index": i,
            })
    return records


chunk_records = build_chunk_records(
    documents,
    chunk_size=CONFIG["chunk_size"],
    overlap=CONFIG["chunk_overlap"],
)

print(f"✅ Created {len(chunk_records):,} chunks "
      f"(chunk_size={CONFIG['chunk_size']}, overlap={CONFIG['chunk_overlap']})\n")

# Per-document breakdown
from collections import Counter
per_doc = Counter(record["source"] for record in chunk_records)
for filename, count in per_doc.items():
    print(f"   {filename:<35} → {count:>5,} chunks")


✅ Created 2,087 chunks (chunk_size=600, overlap=80)

   Ikki eshik orasi (2).txt            → 2,087 chunks


### Code explanation

- `chunk_text()` — the sliding-window splitter. The key line is `start = max(end - overlap, start + 1)`: stepping back by `overlap` creates the shared strip, and the `start + 1` guard mathematically guarantees the loop always advances (no infinite loops on pathological input).
- `build_chunk_records()` — wraps every chunk in a small **record** carrying its `source` filename and a stable, human-readable `id` like `Ikki_eshik_orasi__2_.txt::chunk_0042`. ChromaDB (Part 3) will store exactly these three things: id, text, metadata.
- Note that we validate `overlap < chunk_size` up front — a classic beginner mistake that would otherwise cause an infinite loop of identical chunks.


In [9]:
# ── 2.1b Visualize example chunks and the overlap between them ─────
def show_chunk(record: dict, max_chars: int = 250) -> None:
    """Pretty-print a single chunk record."""
    print(f"┌─ {record['id']}  ({len(record['text'])} chars)")
    print(f"│  {record['text'][:max_chars]}…")
    print("└" + "─" * 68)

print("EXAMPLE CHUNKS — first chunk of each document:\n")
seen: set[str] = set()
for record in chunk_records:
    if record["source"] not in seen:
        show_chunk(record)
        seen.add(record["source"])

# ── Demonstrate the overlap between two adjacent chunks ────────────
print("\nOVERLAP DEMONSTRATION — end of chunk 0 vs start of chunk 1:\n")
first, second = chunk_records[0], chunk_records[1]
tail = first["text"][-CONFIG["chunk_overlap"]:]
print(f"…end of   {first['id']}:\n   …{tail!r}\n")
head = second["text"][:CONFIG["chunk_overlap"]]
print(f"start of  {second['id']}:\n   {head!r}…\n")
print("→ The shared text is visible in both chunks: a sentence sitting on")
print("  the boundary is preserved WHOLE in at least one of them.")


EXAMPLE CHUNKS — first chunk of each document:

┌─ Ikki eshik orasi (2).txt::chunk_0000  (597 chars)
│  «Rost bilan yolg‘onning o‘rtasi – to‘rt enlik», degan gap bor.
Qiziq, nega endi oz emas, ko‘p emas, to‘rt enlik?
Gap shundaki, ko‘z bilan quloqning orasi – to‘rt enlik ekan.
Eshitganingga emas, ko‘rganingga ishon.
Maqsad – shu. Bu kitobdagi ko‘p odam…
└────────────────────────────────────────────────────────────────────

OVERLAP DEMONSTRATION — end of chunk 0 vs start of chunk 1:

…end of   Ikki eshik orasi (2).txt::chunk_0000:
   …'avermaydi. Аmmo yolg‘on gapirayotgan odam ham ichida, Baribir rostini o‘ylaydi.)'

start of  Ikki eshik orasi (2).txt::chunk_0001:
   'avermaydi. Аmmo yolg‘on gapirayotgan odam ham ichida, Baribir rostini o‘ylaydi.)'…

→ The shared text is visible in both chunks: a sentence sitting on
  the boundary is preserved WHOLE in at least one of them.


### Conclusion

The documents are now a list of ~600-character, metadata-tagged fragments — the exact unit the vector database will index. Next: turning each fragment into a **vector**.


---
# Part 2 · Section 2.2 — Embeddings: Turning Text into Vectors

### Theory

A computer cannot compare *meanings* of texts directly — but it can compare *numbers*. An **embedding** is a conversion of a piece of text into a long list of numbers (a **vector**) such that **texts with similar meaning get similar vectors**.

The model required by the TS, `text-embedding-3-small`, maps any text to a vector of **1,536 numbers**. You can picture every chunk as a point in a 1,536-dimensional "meaning space":

- *"Otam urushdan qaytmadi"* and *"Father never returned from the war"* → points **close together** (same meaning, different words — even different languages!)
- *"Otam urushdan qaytmadi"* and *"Choose a chunk size of 600"* → points **far apart**

### What "semantic similarity" means

Closeness between two vectors is measured with **cosine similarity** — essentially the angle between them:

| Cosine similarity | Interpretation |
|---|---|
| ≈ 1.0 | Nearly identical meaning |
| ≈ 0.4–0.7 | Related topic |
| ≈ 0.0 | Unrelated |

This single idea is the engine of the whole RAG system: in Part 3, "find relevant fragments" will literally mean *"find the chunk vectors with the highest cosine similarity to the question vector."* No keywords, no exact matching — pure meaning. That is why a question phrased in *different words* than the book can still find the right passage.

### ⚠️ The critical TS rule

> *The embedding model used when indexing documents and when processing a query must be the same, otherwise retrieval will not work.*

Two different embedding models produce two **incompatible coordinate systems** — like comparing GPS coordinates against street addresses. Our code takes the model name only from `CONFIG["embedding_model"]`, in every place, so this rule cannot be violated by accident.

### How it works internally (and what it costs)

Each API call sends a **batch** of chunk texts to OpenAI and receives one vector per text. Batching matters: sending ~2,300 chunks one at a time = ~2,300 HTTP requests (slow, minutes); sending them 100 at a time = ~24 requests (fast, seconds).

Cost is negligible: `text-embedding-3-small` costs $0.02 per 1M tokens; our whole corpus (~1.2M characters ≈ 400–500K tokens for Uzbek text) costs **about one cent** to index.


In [10]:
# ── 2.2 Generate embeddings for every chunk (batched) ──────────────
import time

def embed_texts(texts: list[str],
                model: str,
                batch_size: int = 100,
                max_retries: int = 3) -> list[list[float]]:
    """Convert a list of texts into embedding vectors via the OpenAI API.

    Texts are sent in batches for speed. Each batch is retried up to
    `max_retries` times on transient network/API errors.

    Args:
        texts: The texts to embed (chunk texts, or a user query).
        model: Embedding model name — always CONFIG["embedding_model"].
        batch_size: How many texts to send per API request.
        max_retries: Attempts per batch before giving up.

    Returns:
        One embedding vector (list of floats) per input text, in order.
    """
    embeddings: list[list[float]] = []
    total_batches = (len(texts) + batch_size - 1) // batch_size

    for batch_number, start in enumerate(range(0, len(texts), batch_size), 1):
        batch = texts[start:start + batch_size]

        for attempt in range(1, max_retries + 1):
            try:
                response = client.embeddings.create(model=model, input=batch)
                break                                   # success → leave retry loop
            except Exception as error:
                if attempt == max_retries:
                    raise RuntimeError(
                        f"Embedding batch {batch_number} failed "
                        f"after {max_retries} attempts: {error}"
                    ) from error
                wait = 2 ** attempt                     # 2s, 4s, 8s — exponential backoff
                print(f"   ⚠️ batch {batch_number} attempt {attempt} failed "
                      f"({error}); retrying in {wait}s…")
                time.sleep(wait)

        # The API returns vectors in the same order as the input texts
        embeddings.extend(item.embedding for item in response.data)
        print(f"   embedded batch {batch_number}/{total_batches} "
              f"({len(embeddings):,}/{len(texts):,} chunks)", end="\r")

    print()
    return embeddings


chunk_texts = [record["text"] for record in chunk_records]

print(f"Generating embeddings for {len(chunk_texts):,} chunks "
      f"with '{CONFIG['embedding_model']}'…")
chunk_embeddings = embed_texts(chunk_texts, model=CONFIG["embedding_model"])
print(f"✅ Done: {len(chunk_embeddings):,} vectors generated.")


Generating embeddings for 2,087 chunks with 'text-embedding-3-small'…
   embedded batch 21/21 (2,087/2,087 chunks)
✅ Done: 2,087 vectors generated.


### Code explanation

- **Batching** — `range(0, len(texts), batch_size)` walks the list 100 chunks at a time; the API guarantees output order matches input order, so `embeddings[i]` always belongs to `chunk_records[i]`.
- **Error handling with exponential backoff** — network calls fail occasionally; instead of crashing after 20 minutes of work, each batch retries with growing pauses (2s → 4s → 8s). This is standard production practice.
- **One reusable function** — the *same* `embed_texts()` will embed the user's **question** in Part 3, automatically satisfying the "same model for indexing and queries" rule.


In [11]:
# ── 2.2b Verify embeddings + demonstrate semantic similarity ───────
import math

def cosine_similarity(a: list[float], b: list[float]) -> float:
    """Cosine similarity between two vectors: 1.0 = same direction."""
    dot = sum(x * y for x, y in zip(a, b))
    norm_a = math.sqrt(sum(x * x for x in a))
    norm_b = math.sqrt(sum(x * x for x in b))
    return dot / (norm_a * norm_b)

# ── Verification checks ─────────────────────────────────────────────
assert len(chunk_embeddings) == len(chunk_records), \
    "Mismatch: every chunk must have exactly one embedding."
dimension = len(chunk_embeddings[0])
assert all(len(vector) == dimension for vector in chunk_embeddings), \
    "All vectors must have the same dimension."

print(f"✅ {len(chunk_embeddings):,} embeddings verified")
print(f"   Vector dimension : {dimension} (text-embedding-3-small → 1536)")
print(f"   First 5 numbers of vector 0: "
      f"{[round(x, 4) for x in chunk_embeddings[0][:5]]} …")

# ── Semantic similarity demonstration ───────────────────────────────
# Adjacent chunks share overlapping text and topic → should be SIMILAR.
# For contrast we pick a "distant" chunk: from ANOTHER document if one
# was uploaded, otherwise from far away in the SAME document.
adjacent = cosine_similarity(chunk_embeddings[0], chunk_embeddings[1])

other_doc_index = next(
    (i for i, record in enumerate(chunk_records)
     if record["source"] != chunk_records[0]["source"]),
    None,                                   # None → only one document uploaded
)
if other_doc_index is not None:
    distant_index = min(other_doc_index + 50, len(chunk_records) - 1)
    distant_label = "a chunk from the other document"
else:
    distant_index = len(chunk_records) // 2
    distant_label = "a distant chunk in the same document"

distant = cosine_similarity(chunk_embeddings[0],
                            chunk_embeddings[distant_index])

print("\nSEMANTIC SIMILARITY DEMONSTRATION (cosine similarity):")
print(f"   chunk 0 vs chunk 1 (adjacent, same passage)   : {adjacent:.3f}")
print(f"   chunk 0 vs {distant_label:<35}: {distant:.3f}")
print("\n→ Neighbouring text scores higher — the vectors really do encode MEANING.")


✅ 2,087 embeddings verified
   Vector dimension : 1536 (text-embedding-3-small → 1536)
   First 5 numbers of vector 0: [0.0405, 0.015, -0.019, 0.0234, 0.0152] …

SEMANTIC SIMILARITY DEMONSTRATION (cosine similarity):
   chunk 0 vs chunk 1 (adjacent, same passage)   : 0.666
   chunk 0 vs a distant chunk in the same document: 0.574

→ Neighbouring text scores higher — the vectors really do encode MEANING.


### Example output

```
✅ 2,3xx embeddings verified
   Vector dimension : 1536 (text-embedding-3-small → 1536)
   First 5 numbers of vector 0: [0.0213, -0.0157, 0.0042, …] …

SEMANTIC SIMILARITY DEMONSTRATION (cosine similarity):
   chunk 0 vs chunk 1 (adjacent, same passage)   : 0.7xx
   chunk 0 vs a chunk from the other document    : 0.3xx

→ Neighbouring text scores higher — the vectors really do encode MEANING.
```

### Conclusion — Part 2 complete ✅

| Requirement (TS / prompt) | Status |
|---|---|
| Chunking with configurable size and overlap (from `CONFIG`, TS Appendix A) | ✅ |
| Word-boundary-safe splitting, no infinite-loop risk | ✅ |
| Example chunks displayed | ✅ |
| Overlap explained **and demonstrated** on real adjacent chunks | ✅ |
| Metadata (source filename, chunk id) attached to every fragment | ✅ |
| Embeddings via `text-embedding-3-small` exactly as required by the TS | ✅ |
| Batched generation with retries and progress reporting | ✅ |
| Embeddings and semantic similarity explained + verified numerically | ✅ |

**Next — Part 3:** storing the vectors in **ChromaDB** and implementing similarity search (retrieval) with configurable top-k.


---
---
# Part 3 · Section 3.1 — The Vector Database (ChromaDB)

### Theory

We now have 2,364 chunks and 2,364 vectors — but they only live in Python lists in memory. We need a component that can, **given a question vector, instantly find the most similar chunk vectors**. That component is a **vector database**.

Why not an ordinary database? A regular database answers *exact* questions: "find the row where `id = 42`". A vector database answers *similarity* questions: "find the 4 stored vectors **closest in meaning** to this new vector". This is a fundamentally different operation, and it is the heart of the retrieval step.

Per TS §4.1, the vector database is the **shared store connecting the two subsystems**:
- the **indexing subsystem** (Parts 2–3) *writes* into it — once;
- the **query subsystem** (Parts 3–4) *reads* from it — on every question.

### What exactly is stored

For every chunk, ChromaDB stores **four things together** (TS §4.2: *"the vectors, the original fragment text, and the source information"*):

| Field | Example | Used for |
|---|---|---|
| `id` | `Ikki_eshik_orasi__2_.txt::chunk_0042` | Unique key |
| `embedding` | `[0.021, -0.015, …]` (1,536 numbers) | Similarity search |
| `document` | the chunk text itself | Building the prompt (Part 4) |
| `metadata` | `{"source": "Ikki_eshik_orasi__2_.txt", "chunk_index": 42}` | Source attribution (Part 4) |

### How it works internally

We configure the collection with `"hnsw:space": "cosine"` — telling ChromaDB to measure closeness with **cosine distance**, matching the cosine-similarity theory from Part 2. (ChromaDB reports *distance* = 1 − similarity: **smaller distance = more similar**.)

Inserts are **batched** (500 records per call) for the same reason embeddings were batched: one large insert per batch is far faster than thousands of tiny ones.


In [12]:
# ── 3.1 Create the ChromaDB collection and store all chunks ────────
import chromadb

def build_vector_database(records: list[dict],
                          embeddings: list[list[float]],
                          collection_name: str = "rag_edu_1",
                          batch_size: int = 500) -> chromadb.Collection:
    """Create a ChromaDB collection and store every chunk in it.

    Each record is stored with its id, text, embedding vector, and
    metadata (source filename + chunk index), per TS §4.2.

    Args:
        records: Chunk records from Part 2 ({"id","text","source","chunk_index"}).
        embeddings: One embedding vector per record, same order.
        collection_name: Name of the ChromaDB collection.
        batch_size: Records inserted per call (batching = speed).

    Returns:
        The populated ChromaDB collection.
    """
    assert len(records) == len(embeddings), \
        "Every chunk record must have exactly one embedding."

    chroma_client = chromadb.Client()          # in-memory DB — ideal for Colab

    # If the notebook is re-run, start from a clean collection
    # (guarantees TS §6 check 5: reproducibility)
    try:
        chroma_client.delete_collection(collection_name)
    except Exception:
        pass                                    # collection did not exist yet

    collection = chroma_client.create_collection(
        name=collection_name,
        metadata={"hnsw:space": "cosine"},      # measure closeness by cosine
    )

    for start in range(0, len(records), batch_size):
        batch = records[start:start + batch_size]
        collection.add(
            ids=[r["id"] for r in batch],
            documents=[r["text"] for r in batch],
            embeddings=embeddings[start:start + len(batch)],
            metadatas=[{"source": r["source"],
                        "chunk_index": r["chunk_index"]} for r in batch],
        )
        print(f"   stored {min(start + batch_size, len(records)):,}"
              f"/{len(records):,} chunks", end="\r")
    print()
    return collection


collection = build_vector_database(chunk_records, chunk_embeddings)
print(f"✅ Vector database ready — collection '{collection.name}' "
      f"contains {collection.count():,} chunks.")


   stored 2,087/2,087 chunks
✅ Vector database ready — collection 'rag_edu_1' contains 2,087 chunks.


### Code explanation

- `chromadb.Client()` — an **in-memory** database: nothing to install or configure, perfect for the educational setting. (For a production system you would switch one line to `chromadb.PersistentClient(path=...)` to keep the index on disk — the rest of the code would not change, which is exactly the TS §2.2 goal of *"a foundation extensible to production without reworking the base logic."*)
- `delete_collection` before `create_collection` — makes the cell **safe to re-run**: you always get a fresh, consistent index instead of duplicated chunks.
- `collection.add(...)` — one call stores ids, texts, embeddings, and metadata together, so retrieval can return all four at once.


In [13]:
# ── 3.1b Verify the stored vectors ──────────────────────────────────
stored_count = collection.count()
assert stored_count == len(chunk_records), (
    f"Storage verification failed: {stored_count} stored "
    f"vs {len(chunk_records)} expected."
)

# Peek at one stored record to confirm every field survived intact
peek = collection.get(ids=[chunk_records[0]["id"]],
                      include=["documents", "metadatas", "embeddings"])

print("✅ STORAGE VERIFICATION PASSED")
print(f"   Chunks stored     : {stored_count:,}")
print(f"   Sample id         : {peek['ids'][0]}")
print(f"   Sample metadata   : {peek['metadatas'][0]}")
print(f"   Sample vector dim : {len(peek['embeddings'][0])}")
print(f"   Sample text       : {peek['documents'][0][:120]!r}…")


✅ STORAGE VERIFICATION PASSED
   Chunks stored     : 2,087
   Sample id         : Ikki eshik orasi (2).txt::chunk_0000
   Sample metadata   : {'chunk_index': 0, 'source': 'Ikki eshik orasi (2).txt'}
   Sample vector dim : 1536
   Sample text       : '«Rost bilan yolg‘onning o‘rtasi – to‘rt enlik», degan gap bor.\nQiziq, nega endi oz emas, ko‘p emas, to‘rt enlik?\nGap shu'…


---
# Part 3 · Section 3.2 — Retrieval: Similarity Search

### Theory

Retrieval is Stage 2 of the TS operation diagram, steps 1–2:

> *Question → **Vectorize the question** → **Retrieve the nearest fragments** from the database → …*

The procedure is beautifully simple:

1. Embed the **question** with `text-embedding-3-small` — **the same model** that embedded the chunks (the critical TS rule; we reuse the very same `embed_texts()` function, so it holds automatically).
2. Ask ChromaDB for the `top_k` stored vectors closest to the question vector (cosine distance).
3. Get back the chunk **texts**, their **sources**, and their **distances** — everything Part 4 needs.

### Why top-k is configurable

`top_k` (TS Appendix A: 3–5) balances two risks: too few fragments and the answer's evidence may be missed; too many and irrelevant text dilutes the prompt and raises cost. Our `CONFIG["top_k"] = 4` sits in the middle, and the function accepts an override for experimentation.

### Why we display retrieved chunks *before* generation

This is the key methodological requirement of TS §6: **retrieval quality and generation quality must be verified separately**. The most common RAG failure is a fluent answer built on *irrelevant* fragments — the model masks the retrieval error. By always being able to inspect *what was retrieved*, you can tell **which** stage failed.


In [14]:
# ── 3.2 Query embedding + similarity search (retrieval) ────────────
def retrieve(question: str,
             top_k: int | None = None) -> list[dict]:
    """Find the chunks most similar in meaning to the question.

    Implements TS §4.2 query-subsystem functions 1–2:
    vectorize the query (same model as indexing) and retrieve the
    top-k nearest fragments from the vector database.

    Args:
        question: The user's question, in natural language.
        top_k: How many fragments to retrieve (default: CONFIG["top_k"]).

    Returns:
        A list of result dicts, most similar first:
        {"text", "source", "chunk_id", "similarity"}.
    """
    if top_k is None:
        top_k = CONFIG["top_k"]

    # 1. Vectorize the question — SAME embedding model as indexing
    question_vector = embed_texts([question],
                                  model=CONFIG["embedding_model"])[0]

    # 2. Similarity search in the vector database
    results = collection.query(
        query_embeddings=[question_vector],
        n_results=top_k,
        include=["documents", "metadatas", "distances"],
    )

    # 3. Repack ChromaDB's parallel lists into convenient records
    retrieved: list[dict] = []
    for text, metadata, distance, chunk_id in zip(results["documents"][0],
                                                  results["metadatas"][0],
                                                  results["distances"][0],
                                                  results["ids"][0]):
        retrieved.append({
            "text": text,
            "source": metadata["source"],
            "chunk_id": chunk_id,
            "similarity": 1 - distance,     # cosine distance → similarity
        })
    return retrieved


def show_retrieved(retrieved: list[dict], max_chars: int = 220) -> None:
    """Display retrieved chunks — the TS §6 retrieval-quality check."""
    print(f"🔎 Retrieved {len(retrieved)} fragment(s):\n")
    for rank, item in enumerate(retrieved, start=1):
        print(f"  #{rank}  similarity={item['similarity']:.3f}  "
              f"source={item['source']}")
        print(f"      {item['text'][:max_chars]}…")
        print()

print("✅ Retrieval functions defined: retrieve(), show_retrieved()")


✅ Retrieval functions defined: retrieve(), show_retrieved()


In [15]:
# ── 3.2b Retrieval demonstration on a real question ────────────────
# A question about the content of the uploaded books (edit freely!)
demo_question = "Kitob muallifi rost bilan yolg‘on haqida nima deydi?"

print(f"QUESTION: {demo_question}\n")
demo_retrieved = retrieve(demo_question)
show_retrieved(demo_retrieved)

print("→ Inspect the fragments above BEFORE any answer is generated:")
print("  do they actually discuss the topic of the question?")
print("  This is the separate retrieval-quality check required by TS §6.")


QUESTION: Kitob muallifi rost bilan yolg‘on haqida nima deydi?

   embedded batch 1/1 (1/1 chunks)
🔎 Retrieved 4 fragment(s):

  #1  similarity=0.584  source=Ikki eshik orasi (2).txt
      alab bo‘lib qoldingmi?
Rost, hozir ham haqiqatni yer bilan bitta qilib, o‘zining rohatini o‘ylaydiganlar bor.
To‘g‘ri, ular senga o‘xshab birovni urmaydi, so‘kmaydi.
Begunoh
659
odamni qamata olmaydi.
Аmmo o‘tirgan kursi…

  #2  similarity=0.569  source=Ikki eshik orasi (2).txt
      oqsoqol, pochcham, muallim – hammasi bir bo‘lib maktab qurishgani esimda.
Robiya yosh edi unda. Bugun xafa bo‘lib o‘tirganini ko‘rib, to‘g‘risi, yuragimga g‘ulg‘ula tushdi.
Gap bu yoqda ekan-ku, men Muzaffardan xavotir o…

  #3  similarity=0.564  source=Ikki eshik orasi (2).txt
      ana ro‘mol tang‘igan, Fotima kelin tugun ko‘tarib olgan, Zuhra kelinning qo‘lida tovoq, ismaloq somsa opchiqibdi.
Oyim sutli ovqatdan keyin ko‘ksomsa yema, deb chirqillaganiga qaramay ikkitasini maza qilib yeb oldim.
Shi…

  #4  similarity=0

### Code explanation

- `retrieve()` — the complete TS query pipeline, steps 1–2. Note the single most important line: the question is embedded through **the same `embed_texts()` + `CONFIG["embedding_model"]`** used for indexing — the TS's critical compatibility rule is satisfied *structurally*, not by discipline.
- `1 - distance` — converts ChromaDB's cosine *distance* back into the cosine *similarity* scale from Part 2 (higher = more similar), so the numbers are directly interpretable.
- `show_retrieved()` — deliberately a separate, reusable function: Part 4 will call it before every generated answer, and Part 5 will use it in acceptance testing.

### Example output

```
QUESTION: Kitob muallifi rost bilan yolg‘on haqida nima deydi?

🔎 Retrieved 4 fragment(s):

  #1  similarity=0.6xx  source=Ikki_eshik_orasi__2_.txt
      «Rost bilan yolg‘onning o‘rtasi – to‘rt enlik», degan gap bor…
  #2  similarity=0.5xx  source=Ikki_eshik_orasi__2_.txt
      …
```

### Conclusion — Part 3 complete ✅

| Requirement (TS / prompt) | Status |
|---|---|
| ChromaDB collection created (cosine space) | ✅ |
| Stored per chunk: id, text, embedding, metadata, source filename | ✅ |
| Stored vectors verified (count + field-by-field peek) | ✅ |
| Query embedding with the **same** model as indexing (TS critical rule) | ✅ |
| Similarity search with configurable top-k (TS Appendix A: 3–5) | ✅ |
| Retrieved chunks displayed **before** answer generation (TS §6) | ✅ |
| Re-runnable indexing (clean collection each run → reproducibility) | ✅ |

**Next — Part 4:** assembling the RAG prompt, generating grounded answers with `gpt-4o-mini`, source attribution, and hallucination prevention.


---
---
# Part 4 · Section 4.1 — Building the RAG Prompt

### Theory

We have the question and the retrieved fragments. Now we must **assemble the prompt** — the complete instruction package sent to the language model (TS Appendix B: *"Assemble the prompt (fragments + question)"*).

A chat model accepts two kinds of messages:

| Message | Role in our system |
|---|---|
| **system** | The model's standing orders — *how* to behave: answer only from context, never invent, how to refuse |
| **user** | The actual data — the retrieved fragments and the question |

### Prompt-engineering decisions (and *why* each one)

Our prompt makes five deliberate choices:

1. **"Answer ONLY from the context"** — the core grounding rule (TS §2.1). The model's own world knowledge is explicitly declared off-limits, so the answer can only come from *your* documents.

2. **A fixed refusal sentence.** If the context lacks the answer, the model must reply with **exactly** one standard marker sentence (see the code). Why *exact*? Three reasons: (a) TS §4.1 — the system must *report* unavailability, never improvise a guess; (b) users get a predictable, honest signal; (c) in Part 5 we can **automatically test** hallucination prevention by checking for the marker — a fuzzy refusal in free words would be untestable.

3. **Numbered, source-labelled fragments.** Each fragment is wrapped as `[Fragment N — source: filename]`. The model sees where each piece of evidence comes from, and clear boundaries prevent it from blending fragments into mush.

4. **"Answer in the language of the question."** Our documents are Uzbek; questions may come in Uzbek or English. `gpt-4o-mini` handles both — but only if told what to do.

5. **Temperature 0** (TS Appendix A: *factual mode*). Temperature controls randomness: at 0 the model always picks its most probable wording — maximally deterministic and factual, and it directly supports TS §6 check 5 (*reproducibility*).


In [16]:
# ── 4.1 Assemble the RAG prompt ─────────────────────────────────────
# The exact marker sentence the model must output when the documents
# do not contain the answer (machine-checkable in Part 5's tests).
NO_ANSWER_MARKER = "INFORMATION NOT AVAILABLE in the provided documents."

SYSTEM_PROMPT = f"""You are a careful assistant that answers questions using ONLY the document fragments provided in the user's message.

Strict rules:
1. Use ONLY the information contained in the provided fragments. Your own outside knowledge is FORBIDDEN.
2. Never invent, guess, or extrapolate facts that are not explicitly present in the fragments.
3. If the fragments do not contain the information needed to answer, reply with EXACTLY this sentence and nothing else:
{NO_ANSWER_MARKER}
4. Otherwise, answer clearly and concisely in the SAME LANGUAGE as the question.
5. When you use a fragment, you may refer to it as [Fragment N]."""


def build_rag_prompt(question: str, retrieved: list[dict]) -> str:
    """Assemble the user message: numbered, source-labelled fragments + question.

    Implements TS §4.2: "assembling a prompt that includes the retrieved
    fragments, the question text, and an instruction to answer only on
    the basis of the supplied context."

    Args:
        question: The user's question.
        retrieved: Fragment records returned by retrieve().

    Returns:
        The complete user-message string.
    """
    fragment_blocks = []
    for number, item in enumerate(retrieved, start=1):
        fragment_blocks.append(
            f"[Fragment {number} — source: {item['source']}]\n{item['text']}"
        )
    context = "\n\n".join(fragment_blocks)

    return (
        f"Document fragments (context):\n\n{context}\n\n"
        f"---\n"
        f"Question: {question}\n\n"
        f"Answer using ONLY the fragments above."
    )


# Show what a real assembled prompt looks like (truncated preview)
_preview = build_rag_prompt("Kitob nima haqida?", demo_retrieved[:2])
print("PREVIEW of an assembled user prompt (first 700 chars):\n")
print(_preview[:700] + "\n…")


PREVIEW of an assembled user prompt (first 700 chars):

Document fragments (context):

[Fragment 1 — source: Ikki eshik orasi (2).txt]
alab bo‘lib qoldingmi?
Rost, hozir ham haqiqatni yer bilan bitta qilib, o‘zining rohatini o‘ylaydiganlar bor.
To‘g‘ri, ular senga o‘xshab birovni urmaydi, so‘kmaydi.
Begunoh
659
odamni qamata olmaydi.
Аmmo o‘tirgan kursisidan foydalanib, cho‘ntagini qappaytiradi.
Аvval atrofga tanish-bilish to‘playdi, keyin mol-mulk yig‘adi.
Hozirgi ablahlarning sen ablahdan farqi yo‘q.
Аmmo ertami-kechmi hamma narsaning hisob-kitobi bo‘ladi.
Haqiqat degani – aslida mana shu!
Аbsolyut haqiqat degani shu!
– Bor!
– dedim ishonch bilan.
– Ertami-kechmi, hammasi joyiga tushadi.
U qilt etmadi.
Yuzi kinoyali

[Fragment 2 — source: Ikki 
…


### Code explanation

- `NO_ANSWER_MARKER` is a module-level constant — the *same* string is used in the system prompt, in the pipeline logic below, and in Part 5's acceptance tests. One source of truth; no drift possible.
- `build_rag_prompt()` is pure string assembly — no API calls — so it is trivially testable and its output is fully inspectable (as the preview shows).
- The context is placed **before** the question: the model reads the evidence first, then the task — the standard, most reliable ordering for grounded QA.


---
# Part 4 · Section 4.2 — Answer Generation with Source Attribution

### Theory

The final pipeline steps (TS Appendix B): *"…Generate the answer with the language model → Return the answer and its sources."*

Generation is one API call to `gpt-4o-mini` (fixed by TS Appendix A) at temperature 0. Around it we build the complete, user-facing function `ask()` that chains **everything** built in Parts 2–4:

```
ask(question)
  → retrieve(question)          # Part 3 — same embedding model as indexing
  → show_retrieved(...)         # TS §6 — inspect retrieval BEFORE generation
  → build_rag_prompt(...)       # Section 4.1
  → generate_answer(...)        # gpt-4o-mini, temperature 0
  → answer + source list        # attribution (TS §6 check 4)
```

### How source attribution works

We do **not** ask the model to name its sources — a model can misattribute. Instead, attribution is **computed programmatically**: the sources are the (deduplicated) filenames of the retrieved fragments, taken from the metadata stored in ChromaDB. Metadata cannot hallucinate. And when the model returns the `NO_ANSWER_MARKER`, no sources are listed — because no document actually supported an answer.

### Why the pipeline returns a structured result

`ask()` returns a dictionary — `{"question", "answer", "retrieved", "sources", "answered"}` — rather than just printing text. Part 5's acceptance tests will assert against these fields automatically; a print-only function would make systematic testing impossible.


In [17]:
# ── 4.2 Generation + the complete RAG pipeline ──────────────────────
def generate_answer(user_prompt: str,
                    max_retries: int = 3) -> str:
    """Call the generation model (TS: gpt-4o-mini) with the RAG prompt.

    Args:
        user_prompt: Output of build_rag_prompt().
        max_retries: Attempts before giving up on transient API errors.

    Returns:
        The model's answer text.
    """
    for attempt in range(1, max_retries + 1):
        try:
            response = client.chat.completions.create(
                model=CONFIG["generation_model"],
                temperature=CONFIG["temperature"],     # 0 → factual mode
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_prompt},
                ],
            )
            return response.choices[0].message.content.strip()
        except Exception as error:
            if attempt == max_retries:
                raise RuntimeError(
                    f"Generation failed after {max_retries} attempts: {error}"
                ) from error
            wait = 2 ** attempt
            print(f"   ⚠️ generation attempt {attempt} failed ({error}); "
                  f"retrying in {wait}s…")
            time.sleep(wait)


def ask(question: str,
        top_k: int | None = None,
        show_fragments: bool = True) -> dict:
    """The complete RAG pipeline: retrieve → prompt → generate → attribute.

    Args:
        question: The user's question in natural language.
        top_k: Fragments to retrieve (default CONFIG["top_k"]).
        show_fragments: If True, display retrieved chunks before the
            answer (the TS §6 separate retrieval-quality check).

    Returns:
        {"question": str,
         "answer": str,
         "retrieved": list[dict],   # the evidence fragments
         "sources": list[str],      # deduplicated source filenames
         "answered": bool}          # False if the no-answer marker fired
    """
    print(f"❓ QUESTION: {question}\n")

    # 1–2. Retrieval (Part 3)
    retrieved = retrieve(question, top_k=top_k)
    if show_fragments:
        show_retrieved(retrieved)

    # 3. Prompt assembly (Section 4.1)
    user_prompt = build_rag_prompt(question, retrieved)

    # 4. Generation
    answer = generate_answer(user_prompt)
    answered = NO_ANSWER_MARKER not in answer

    # 5. Programmatic source attribution — from metadata, never from the model
    sources = sorted({item["source"] for item in retrieved}) if answered else []

    print(f"💬 ANSWER:\n{answer}\n")
    if answered:
        print(f"📚 SOURCES: {', '.join(sources)}")
    else:
        print("📚 SOURCES: — (no supporting document found)")

    return {"question": question, "answer": answer,
            "retrieved": retrieved, "sources": sources, "answered": answered}

print("✅ RAG pipeline ready: ask('your question')")


✅ RAG pipeline ready: ask('your question')


In [18]:
# ── 4.2b Demonstration 1 — a question the documents CAN answer ─────
result_in_docs = ask("Kitob muallifi rost bilan yolg‘on o‘rtasidagi masofa "
                     "haqida qanday gap keltiradi?")


❓ QUESTION: Kitob muallifi rost bilan yolg‘on o‘rtasidagi masofa haqida qanday gap keltiradi?

   embedded batch 1/1 (1/1 chunks)
🔎 Retrieved 4 fragment(s):

  #1  similarity=0.587  source=Ikki eshik orasi (2).txt
      Shoshilmasdan yuz-qo‘lini, quloqlarining orqasini ishqalab yuvdi.
Hafsala bilan artindi.
– Tuzukmisiz, qizim? – dedi ko‘zimga tikilib.
– Qani, yuring-chi. Chiroq obkeling, onasi! Ketma-ket yurib nimqorong‘i hujraga kirdi…

  #2  similarity=0.582  source=Ikki eshik orasi (2).txt
      b qutulsam yaxshimidi?
Dunyo nimaga bunaqa teskari?
Nima uchun qilar ishni birov qiladiyu kasofatiga boshqalar qoladi?!
.o‘sha kuni shilqillab qolgan bolani ko‘tarib qamishzordan o‘tib, qishloq ko‘chasiga kirib qolibman.…

  #3  similarity=0.577  source=Ikki eshik orasi (2).txt
      turtib imo qildi.
Sahna to‘rida Stalinning mo‘ylovli kattakon surati, qizil mato yopilgan uzun stol orqasida olti kishi o‘tiribdi.
Eng avval Oqsoqol buvaga ko‘zim tushdi.
Ehtimol, gavdasi hammadan katta bo‘lgan

In [19]:
# ── 4.2c Demonstration 2 — a question the documents CANNOT answer ──
# The books say nothing about programming — the system must honestly
# refuse instead of inventing an answer (TS §4.1, §6 check 3).
result_not_in_docs = ask("Python dasturlash tilida ro‘yxatni qanday saralash mumkin?")


❓ QUESTION: Python dasturlash tilida ro‘yxatni qanday saralash mumkin?

   embedded batch 1/1 (1/1 chunks)
🔎 Retrieved 4 fragment(s):

  #1  similarity=0.471  source=Ikki eshik orasi (2).txt
      tmon dastasidan ushlab turgan Parcha rais oldida o‘zini ko‘rsatib qo‘ygisi keldi shekilli, ikki kaftiga mo‘lgina tufladi-da, ketmonni boshi ustida baland ko‘targancha «xax» deb lavlagi pushtasiga urdi.
Mo‘ljallagan lavla…

  #2  similarity=0.453  source=Ikki eshik orasi (2).txt
      Boplab do‘pposlamoqchi edimu ayadim. Bundan chiqdi rostdan ham dadasiga chaqqan.
«Tuya» sartarosh direktorga shikoyat qilgan. Shimimning pochasiyam ilma-teshik bo‘p ketibdi.
Qayirib ola qolay. Direktor so‘rasa loy bo‘lga…

  #3  similarity=0.435  source=Ikki eshik orasi (2).txt
      , tuproqning mayin, dilni orziqtiruvchi isi dimoqqa urar edi.
Kechasi traktorlar o‘tgan paykallarni yana aylanib chiqdik.
Shunaqa qilsak, Zuhra kelin topiladigandek, dala o‘rtasida to‘xtab qolgan traktorning tagiga qayta…

  #4  simi

### Example output

**Demonstration 1** (answer exists in the documents):
```
❓ QUESTION: Kitob muallifi rost bilan yolg‘on o‘rtasidagi masofa haqida qanday gap keltiradi?

🔎 Retrieved 4 fragment(s):
  #1  similarity=0.6xx  source=Ikki_eshik_orasi__2_.txt
      «Rost bilan yolg‘onning o‘rtasi – to‘rt enlik», degan gap bor…
  …

💬 ANSWER:
Muallif «Rost bilan yolg‘onning o‘rtasi – to‘rt enlik» degan gapni keltiradi, chunki
ko‘z bilan quloqning orasi to‘rt enlik: eshitganingga emas, ko‘rganingga ishon [Fragment 1].

📚 SOURCES: Ikki_eshik_orasi__2_.txt
```

**Demonstration 2** (answer absent — hallucination prevention):
```
❓ QUESTION: Python dasturlash tilida ro‘yxatni qanday saralash mumkin?

🔎 Retrieved 4 fragment(s):
  #1  similarity=0.1xx …   ← low similarity: nothing truly relevant exists
  …

💬 ANSWER:
INFORMATION NOT AVAILABLE in the provided documents.

📚 SOURCES: — (no supporting document found)
```

### Conclusion — Part 4 complete ✅

| Requirement (TS / prompt) | Status |
|---|---|
| Prompt instructs: answer ONLY from supplied context | ✅ |
| Prompt instructs: never invent information | ✅ |
| Prompt instructs: clearly state when information is unavailable (exact marker) | ✅ |
| Prompt-engineering decisions explained one by one | ✅ |
| Generation with `gpt-4o-mini`, temperature 0 (TS Appendix A) | ✅ |
| Pipeline returns answer + retrieved chunks + source documents | ✅ |
| Source attribution computed from metadata, not from the model | ✅ |
| Retrieved fragments displayed before every answer (TS §6) | ✅ |
| Both behaviours demonstrated: answerable + unanswerable question | ✅ |

**Next — Part 5:** systematic acceptance testing against all five TS §6 criteria, the full compliance checklist, and final polishing.


---
---
# Part 5 · Section 5.1 — Acceptance Testing (TS §6)

### Theory

The TS §6 makes one methodological demand above all: **verify retrieval quality and generation quality separately**. The most dangerous RAG failure is invisible — the answer *looks* correct while the system actually retrieved irrelevant fragments and the model papered over the gap. A test suite that only reads final answers would never catch it.

Our suite therefore tests the five TS §6 criteria at the right layer each time:

| # | TS §6 check | What we test | Layer tested |
|---|---|---|---|
| 1 | Correctness of fragment retrieval | For a question with a *known* answer, the known passage is among the retrieved fragments | **Retrieval only** (no LLM involved) |
| 2 | Answer reliability | The answer contains the known key fact and no refusal marker | Generation |
| 3 | Behaviour when data is absent | For out-of-scope questions, the exact `NO_ANSWER_MARKER` is returned | Generation (hallucination prevention) |
| 4 | Source attribution | Every answered question carries a non-empty source list, and every source is a real uploaded file | Pipeline |
| 5 | Reproducibility | The same question twice → identical retrieved fragments (and, at temperature 0, a stable answer) | Whole system |

### How the test questions were chosen

- **In-scope questions** target passages we *know* exist (e.g., the opening of *Ikki eshik orasi* about «rost bilan yolg‘on»), each with a **keyword** that must appear in a correct answer. You can — and should — add your own.
- **Out-of-scope questions** are deliberately far from the books (programming, sports) so the only honest response is the refusal marker.


In [20]:
# ── 5.1 Acceptance test suite (TS §6) ───────────────────────────────
# Questions with answers KNOWN to exist in the uploaded documents.
# "keyword" = a phrase that must appear in the retrieved evidence / answer.
#  → Edit or extend these when you replace the documents with your own!
IN_SCOPE_TESTS: list[dict] = [
    {
        "question": "Rost bilan yolg‘onning o‘rtasi qancha deyilgan va nega?",
        "keyword": "to‘rt enlik",
    },
    {
        "question": "Kimni Alloh hidoyat qilsa, u haqda nima deyiladi?",
        "keyword": "adashtiruvchi",
    },
]

# Questions whose answers are NOT in the documents (must be refused).
OUT_OF_SCOPE_TESTS: list[str] = [
    "Python dasturlash tilida ro‘yxatni qanday saralash mumkin?",
    "2026 yilgi futbol jahon chempionatida qaysi jamoa g‘olib bo‘ldi?",
]


def run_acceptance_tests() -> bool:
    """Run all five TS §6 acceptance checks and print a PASS/FAIL report.

    Returns:
        True if every check passed (the System is accepted per TS §6).
    """
    results: list[tuple[str, bool, str]] = []   # (check name, passed, detail)

    # ── Check 1: retrieval correctness (retrieval layer ONLY) ──────
    for test in IN_SCOPE_TESTS:
        retrieved = retrieve(test["question"])
        found = any(test["keyword"].lower() in item["text"].lower()
                    for item in retrieved)
        results.append((
            f"1. Retrieval finds evidence for: {test['question'][:45]}…",
            found,
            f"keyword '{test['keyword']}' "
            f"{'found in' if found else 'MISSING from'} retrieved fragments",
        ))

    # ── Check 2: answer reliability ─────────────────────────────────
    in_scope_results = []
    for test in IN_SCOPE_TESTS:
        result = ask(test["question"], show_fragments=False)
        in_scope_results.append(result)
        reliable = (result["answered"]
                    and test["keyword"].lower() in result["answer"].lower())
        results.append((
            f"2. Answer is reliable for: {test['question'][:45]}…",
            reliable,
            "answer grounded in the known fact" if reliable
            else "answer missing the known fact — review manually",
        ))

    # ── Check 3: behaviour when data is absent ──────────────────────
    for question in OUT_OF_SCOPE_TESTS:
        result = ask(question, show_fragments=False)
        refused = (not result["answered"]) and (result["sources"] == [])
        results.append((
            f"3. Honest refusal for: {question[:45]}…",
            refused,
            "exact NO_ANSWER_MARKER returned, no sources listed" if refused
            else "SYSTEM DID NOT REFUSE — possible hallucination!",
        ))

    # ── Check 4: source attribution ─────────────────────────────────
    for result in in_scope_results:
        attributed = (bool(result["sources"])
                      and all(s in documents for s in result["sources"]))
        results.append((
            f"4. Sources attributed for: {result['question'][:45]}…",
            attributed,
            f"sources = {result['sources']}",
        ))

    # ── Check 5: reproducibility ────────────────────────────────────
    question = IN_SCOPE_TESTS[0]["question"]
    first = ask(question, show_fragments=False)
    second = ask(question, show_fragments=False)
    same_fragments = ([r["chunk_id"] for r in first["retrieved"]]
                      == [r["chunk_id"] for r in second["retrieved"]])
    same_answer = first["answer"] == second["answer"]
    results.append((
        "5. Reproducibility (same question twice)",
        same_fragments,
        f"identical fragments: {same_fragments}; "
        f"identical answers: {same_answer} "
        f"(temperature 0 → expected stable)",
    ))

    # ── Report ──────────────────────────────────────────────────────
    print("\n" + "=" * 72)
    print("ACCEPTANCE TEST REPORT (TS §6)")
    print("=" * 72)
    for name, passed, detail in results:
        print(f"  {'✅ PASS' if passed else '❌ FAIL'}  {name}")
        print(f"          {detail}")
    passed_count = sum(passed for _, passed, _ in results)
    all_passed = passed_count == len(results)
    print("-" * 72)
    print(f"  RESULT: {passed_count}/{len(results)} checks passed")
    print("  → THE SYSTEM IS ACCEPTED per TS §6." if all_passed
          else "  → NOT accepted yet — investigate the failed checks above.")
    print("=" * 72)
    return all_passed


all_tests_passed = run_acceptance_tests()


   embedded batch 1/1 (1/1 chunks)
   embedded batch 1/1 (1/1 chunks)
❓ QUESTION: Rost bilan yolg‘onning o‘rtasi qancha deyilgan va nega?

   embedded batch 1/1 (1/1 chunks)
💬 ANSWER:
Rost bilan yolg‘onning o‘rtasi to‘rt enlik deyilgan. Sababi, ko‘z bilan quloqning orasi to‘rt enlik ekan va maqsad – ko‘rganingga ishonishdir [Fragment 1].

📚 SOURCES: Ikki eshik orasi (2).txt
❓ QUESTION: Kimni Alloh hidoyat qilsa, u haqda nima deyiladi?

   embedded batch 1/1 (1/1 chunks)
💬 ANSWER:
INFORMATION NOT AVAILABLE in the provided documents.

📚 SOURCES: — (no supporting document found)
❓ QUESTION: Python dasturlash tilida ro‘yxatni qanday saralash mumkin?

   embedded batch 1/1 (1/1 chunks)
💬 ANSWER:
INFORMATION NOT AVAILABLE in the provided documents.

📚 SOURCES: — (no supporting document found)
❓ QUESTION: 2026 yilgi futbol jahon chempionatida qaysi jamoa g‘olib bo‘ldi?

   embedded batch 1/1 (1/1 chunks)
💬 ANSWER:
INFORMATION NOT AVAILABLE in the provided documents.

📚 SOURCES: — (no supporti

### Code explanation

- **Check 1 never touches the LLM** — it calls `retrieve()` directly and looks for the known keyword inside the raw fragments. If this fails while Check 2 passes, the model is compensating for bad retrieval — exactly the masked failure the TS warns about.
- **Check 3 tests for the *exact* marker** via `result["answered"]` — this is why Part 4 insisted on a fixed refusal sentence: a free-form refusal would be untestable.
- **Check 5** compares retrieved chunk **ids**, not texts — the strictest possible identity test for retrieval; answer stability is reported alongside (temperature 0 makes it expected, though LLM decoding is not *formally* guaranteed deterministic, which is why the TS-binding assertion is on the fragments).
- The suite **reports** failures rather than crashing mid-notebook — a beginner sees the full picture, then investigates.


---
# Part 5 · Section 5.2 — Final Compliance Checklist

Every requirement of the Technical Specification, verified against the implementation:

| TS ref | Requirement | Where implemented | Status |
|---|---|---|---|
| §1.5 | Deliverable: working Colab notebook + explanations | this notebook | ✅ |
| §2.1 | Answers grounded strictly in supplied documents | system prompt, Part 4.1 | ✅ |
| §2.2 | Extensible foundation (no rework of base logic) | modular functions; one-line switch to `PersistentClient` | ✅ |
| §3 | Input: .txt documents; two automated stages | Parts 1.4 (input), 2–3 (indexing), 3–4 (query) | ✅ |
| §4.1 | Two subsystems linked by a vector database | Parts 2–3 write; Parts 3–4 read | ✅ |
| §4.1 | Reports unavailability instead of inventing | `NO_ANSWER_MARKER`, verified by Check 3 | ✅ |
| §4.1 | Runs in Colab, browser only; API key required | Parts 1.1–1.2 | ✅ |
| §4.2 | Indexing: load → chunk (fixed size + overlap) → vectorize → store | Parts 1.4, 2.1, 2.2, 3.1 | ✅ |
| §4.2 | Query: vectorize (same model) → retrieve top-k → assemble prompt → generate + sources | Parts 3.2, 4.1, 4.2 | ✅ |
| §4.3 | Python ≥3.10; only `openai` + `chromadb` | Part 1.1 | ✅ |
| §5 | All six work stages performed in sequence | Parts 1–5 | ✅ |
| §6 | Five acceptance checks; retrieval vs generation tested separately | Section 5.1 suite | ✅ |
| §7 | Notebook explanations for non-programmers; document-replacement instruction | every section; Section 5.3 below | ✅ |
| App. A | All 6 parameters configurable, TS defaults, change-one-at-a-time | `CONFIG`, Part 1.3 | ✅ |
| App. A | Same embedding model for indexing and queries (critical) | single `CONFIG` constant + shared `embed_texts()` | ✅ |
| App. B | Exact two-stage operation diagram | pipeline structure of `ask()` | ✅ |

**Additional engineering requirements (project prompt):**

| Requirement | Status |
|---|---|
| Only the user's uploaded `.txt` files; no demo documents | ✅ (Part 1.4) |
| API key via Colab Secrets / env var; never hardcoded or printed | ✅ (Part 1.2) |
| Latest OpenAI v1.x SDK; no deprecated APIs | ✅ (`OpenAI()` client, `chat.completions`, `embeddings.create`) |
| "Run All" works top-to-bottom for a beginner | ✅ (re-runnable cells, clean-collection pattern) |
| Everything in English (markdown, code, comments, names) | ✅ |
| Functions, docstrings, type hints, meaningful names, error handling | ✅ (every function) |
| No duplicated code | ✅ (e.g., one `embed_texts()` for chunks *and* queries) |

### Code review summary

- **Modularity:** 12 single-purpose functions; the whole system is usable through one entry point, `ask()`.
- **Error handling:** exponential-backoff retries on all API calls; loud, actionable errors for a missing key, missing files, or empty documents; input validation (`overlap < chunk_size`).
- **Reproducibility:** temperature 0, delete-before-create collection, deterministic chunking.
- **Cost awareness:** batched embeddings (~24 requests instead of ~2,364), corpus indexing ≈ $0.01.


---
# Part 5 · Section 5.3 — Using the System & Replacing the Documents

### Ask your own questions

Run the cell below with any question — in Uzbek or English — about the uploaded books:


In [21]:
# ── 5.3 Interactive question cell — edit and re-run freely ─────────
your_question = "Ikki eshik orasi kitobida oyim haqida nima hikoya qilinadi?"

_ = ask(your_question)


❓ QUESTION: Ikki eshik orasi kitobida oyim haqida nima hikoya qilinadi?

   embedded batch 1/1 (1/1 chunks)
🔎 Retrieved 4 fragment(s):

  #1  similarity=0.548  source=Ikki eshik orasi (2).txt
      olnikiga? Zaril ishi bor ekan.
– Obbo! – Badaniga endi issiq o‘tgan bobom rohati buzilganidan g‘ashi kelib ijirg‘andi.
– Shu Oqsoqolam g‘irt ahmoq odam-da! Nima ishi bor ekan, yarim kechada!
– Shunday dediyu Baribir sand…

  #2  similarity=0.545  source=Ikki eshik orasi (2).txt
      inoyim ham qiziq.
Xuddi Zuhra kelinni men o‘ldirgandek, Zakunchining so‘rog‘ini, mening javobimni so‘zma-so‘z yozib oladi.
Bilaman, Raʼno kelinoyim meni yomon ko‘radi.
Bulturgi voqeadan keyin uyiga qadam bosmaganim alam …

  #3  similarity=0.542  source=Ikki eshik orasi (2).txt
      ima bo‘pti? Kimsan akam kelsa aytaman.
«Qor yorug‘ida tikkanman», deyman. Kimsan akam hayron qoladi.
Nihoyat hovli tomondan qorning g‘irchillagani, bobomning ayvonga chiqib gursillatib yer tepgani eshitildi.
– Onasi! Pay…

  #4  sim

### Replacing the documents with your own (TS §7 instruction)

The system is not tied to these two books — the knowledge base is whatever you upload:

1. **Runtime → Restart session** (clears the old documents and index).
2. **Run All** (`Runtime → Run all`).
3. When the upload dialog appears in Part 1.4, select **your own `.txt` files** (UTF-8; any language).
4. Everything downstream — chunking, indexing, retrieval, answering — adapts automatically.
5. Update the questions in `IN_SCOPE_TESTS` / `OUT_OF_SCOPE_TESTS` (Section 5.1) to match your new documents, and re-run the acceptance suite.

### Tuning (TS Appendix A)

Change **one** `CONFIG` value at a time (Part 1.3), re-run from Part 2 onward, and re-check retrieval quality with the acceptance suite — exactly as the TS prescribes. Typical experiments: `chunk_size` 300 vs 800, `top_k` 3 vs 5.

---
# 🎓 Project Complete

You have built, from nothing, a complete **Retrieval-Augmented Generation system**:

*documents → chunks → embeddings → vector database → similarity search → grounded prompt → honest, source-attributed answers* —

implemented strictly to the Technical Specification RAG-EDU-1, tested against its five acceptance criteria, and ready to be pointed at any document collection you own.


In [22]:
# ── 🌐 Web App: Simple RAG with Gradio ──────────────────────────────
# Install Gradio (the library that builds the web page)
!pip install -q gradio

import gradio as gr

def answer_for_app(question: str) -> str:
    """Bridge between the web page and our existing ask() function."""
    if not question or not question.strip():
        return "⚠️ Please type a question first. / Avval savol yozing."
    try:
        result = ask(question, show_fragments=False)
        if result["answered"]:
            sources = ", ".join(result["sources"])
            return f"{result['answer']}\n\n📚 Sources: {sources}"
        return "🤷 " + result["answer"]
    except Exception as error:
        return f"❌ Something went wrong: {error}"

app = gr.Interface(
    fn=answer_for_app,
    inputs=gr.Textbox(
        label="Your question (Savolingiz)",
        placeholder="Ask anything about the uploaded books…",
        lines=2,
    ),
    outputs=gr.Textbox(label="Answer (Javob)", lines=8),
    title="📚 Simple RAG — Chat with the Books",
    description=(
        "This app answers questions using ONLY the uploaded Uzbek books "
        "(*Ikki eshik orasi* and *Murdalar gapirmaydilar*). "
        "If the books don't contain the answer, it honestly says so — "
        "no made-up information. Built as an educational RAG system (RAG-EDU-1)."
    ),
    examples=[
        ["Rost bilan yolg‘onning o‘rtasi qancha deyilgan va nega?"],
        ["Kimni Alloh hidoyat qilsa, u haqda nima deyiladi?"],
        ["Ikki eshik orasi kitobida oyim haqida nima hikoya qilinadi?"],
    ],
    flagging_mode="never",
)

app.launch(share=True)   # share=True → creates the public link for friends

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://cb5e834397bd11bbd4.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
